## Agent Traces to Supervised FineTuning

This demo shows how Agent Traces from a tool-calling Hosted Agent can be used to perform Supervised FineTuning (SFT) on an AzureOpenAI model. 

In [4]:
%pip install azure_ai_projects-2.2.0-py3-none-any.whl

In [44]:
import os
from dotenv import load_dotenv

load_dotenv()

FOUNDRY_PROJECT_ID = os.environ["FOUNDRY_PROJECT_ID"]
APPLICATION_INSIGHTS_ID = os.environ["APPLICATION_INSIGHTS_ID"]

Fetch Foundry Project's System-Assigned Managed Identity Principal ID

In [ ]:
!az resource show --ids {FOUNDRY_PROJECT_ID} --query identity.principalId -o tsv

Copy the above Principal ID in the below command:

In [ ]:
!az role assignment create --assignee "PRINCIPAL_ID" --role "Log Analytics Reader" --scope {APPLICATION_INSIGHTS_ID}

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential
)

In [8]:
page = project_client.beta.datasets.list_generation_jobs()
jobs = list(page)
print(jobs[0].id, jobs[0].inputs.name)

datagen-0fdfc7f078bf4229b60384ccc3b77d07 sft-from-traces-20260507-040023


In [9]:
job = project_client.beta.datasets.get_generation_job(jobs[0].id)
print(job.id, job.inputs.name)

datagen-0fdfc7f078bf4229b60384ccc3b77d07 sft-from-traces-20260507-040023


In [31]:
from azure.ai.projects.models import (
    DataGenerationJob,
    DataGenerationJobInputs,
    DataGenerationJobScenario,
    TracesDataGenerationJobOptions,
    TracesDataGenerationJobSource,
)

# HACK: reorder _data dict so "type" is serialized first. Will be fixed service side.
def _put_type_first(model):
    if hasattr(model, "_data") and "type" in model._data:
        model._data = {"type": model._data["type"], **{k: v for k, v in model._data.items() if k != "type"}}

options = TracesDataGenerationJobOptions(
    max_samples=50,
    train_split=0.8
)
_put_type_first(options)

source = TracesDataGenerationJobSource(
    agent_name="aprilk-tracebed-af-responses",
    start_time=1778070134
)
_put_type_first(source)

job = DataGenerationJob(
    inputs=DataGenerationJobInputs(
        name="sft_traces_data",
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        options=options,
        sources=[source]
    )
)

job = project_client.beta.datasets.create_generation_job(job)

In [12]:
import time
from IPython.display import clear_output
from azure.ai.projects.models import JobStatus

while job.status not in [JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.CANCELLED]:
    job = project_client.beta.datasets.get_generation_job(job.id)
    clear_output(wait=True)
    print(f"Job status: {job.status}")
    time.sleep(10)

In [ ]:
from azure.ai.projects.models import DataGenerationJobOutputType

for output in job.result.outputs:
    assert output.type == DataGenerationJobOutputType.FILE

    print(f"Output {output.type}: File ID: {output.id}, Filename: {output.filename}")

Output file: File ID: file-68a0d0cd4a744cc6b1198279847bc5a8, Filename: None
Output file: File ID: file-ff70b44e3e5d485fa1ea2035a74ab974, Filename: None


In [20]:
file_ids = [output.id for output in job.result.outputs if output.type == DataGenerationJobOutputType.FILE]

training_file = file_ids[0]
validation_file = file_ids[1] if len(file_ids) > 1 else None

In [ ]:
simpleqna_job = project_client.beta.datasets.get_generation_job("datagen-d174ba525b5b41d08c95769f817835bf")

In [26]:
simpleqna_job.result.token_usage.prompt_tokens

4919

In [28]:
project_client.beta.datasets.delete_generation_job(simpleqna_job.id)

In [ ]:
project_client.beta.datasets.cancel_generation_job(job.id)